In [2]:
!pip install transformers==4.51.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 39.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 82.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import transformers
print("transformers:", transformers.__version__)

In [ ]:
!pip install huggingface-hub==0.34.4
from huggingface_hub import login

login()

In [ ]:
!pip install -U bitsandbytes


In [ ]:
!pip uninstall -y torch torchvision torchaudio

In [ ]:
!pip install --upgrade torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121


In [ ]:
import torch
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import argparse
import json
import random
import os, json, re, random, math
from collections import Counter, defaultdict
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    AutoModelForSeq2SeqLM
)
from peft import LoraConfig, get_peft_model, PeftModel

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# train_one_trial.py
# Run ONE hyperparameter trial in its own process.
# Usage example:
#   python train_one_trial.py --lr 1e-4 --r 8 --alpha 16 --dropout 0.05 --epochs 1 --out runs/trial1 --max_val_samples 30
%%writefile train_one_trial.py

import os, json, re, random, argparse, gc
from collections import Counter
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, PeftModel


# -----------------------------
# Data helpers (EmoWOZ -> samples)
# -----------------------------
def extract_emotion_id(emotion_list):
    if not emotion_list:
        return None
    for e in emotion_list:
        if isinstance(e, dict) and "emotion" in e:
            return int(e["emotion"])
    ann = []
    for e in emotion_list:
        if isinstance(e, dict) and "annotation" in e:
            ann.append(int(e["annotation"]))
    if ann:
        return Counter(ann).most_common(1)[0][0]
    return None


def dialog_act_to_facts(dialog_act):
    facts = []
    if not dialog_act:
        return ""
    for act, pairs in dialog_act.items():
        for p in pairs:
            if isinstance(p, (list, tuple)) and len(p) >= 2:
                slot, val = p[0], p[1]
                facts.append(f"{act}.{slot} = {val}")
    return "\n".join(facts)


def build_emowoz_samples(emowoz_json, max_dialogs=None):
    samples = []
    dialog_ids = list(emowoz_json.keys())
    if max_dialogs:
        dialog_ids = dialog_ids[:max_dialogs]

    for did in dialog_ids:
        log = emowoz_json[did]["log"]
        history = []
        accumulated_facts = []
        last_user_emotion = None

        for turn in log:
            text = str(turn.get("text", "")).strip()
            dialog_act = turn.get("dialog_act", {}) or {}
            emo_list = turn.get("emotion", []) or []
            if not text:
                continue

            has_act = bool(dialog_act)
            has_emo = bool(emo_list)

            if has_act:
                role = "assistant"
            elif has_emo:
                role = "user"
            else:
                role = "assistant" if (len(history) > 0 and history[-1].startswith("User:")) else "user"

            if role == "user":
                last_user_emotion = extract_emotion_id(emo_list)
                history.append(f"User: {text}")
            else:
                new_facts = dialog_act_to_facts(dialog_act)
                if new_facts:
                    accumulated_facts.append(new_facts)

                sample = f"""### Instruction:
You are an emotionally intelligent, professional task-oriented dialogue agent.
You must be empathetic and calm, and you must NOT invent facts.
Use only the provided facts when stating addresses, phone numbers, times, prices, etc.

### User Emotion:
{str(last_user_emotion) if last_user_emotion is not None else "unknown"}

### Provided Facts:
{chr(10).join(accumulated_facts) if accumulated_facts else "none"}

### Context:
{chr(10).join(history)}

### Response:
{text}
"""
                samples.append(sample.strip())
                history.append(f"Assistant: {text}")
    return samples


def strip_gold_response(sample_text):
    if "### Response:" not in sample_text:
        return sample_text.strip()
    before = sample_text.split("### Response:")[0].strip()
    return before + "\n\n### Response:\n"


# -----------------------------
# Judge (Flan-T5)
# -----------------------------
def make_judge(judge_name, device):
    judge_tokenizer = AutoTokenizer.from_pretrained(judge_name)
    judge_model = AutoModelForSeq2SeqLM.from_pretrained(judge_name).to(device)
    judge_model.eval()
    return judge_tokenizer, judge_model


def judge_score(judge_tokenizer, judge_model, context_block, candidate_response):
    prompt = f"""
You are a strict evaluator for a task-oriented dialogue agent.

You will score the agent response using 3 criteria, each 0 to 5:
1) Relevance: answers the user's request based on the context.
2) Empathy: matches the user's emotion and is calm/professional.
3) Grounding: does NOT invent facts; uses only Provided Facts for specific details.

Return ONLY in this exact format:
Relevance=<int> Empathy=<int> Grounding=<int>

### INPUT
{context_block}

### AGENT RESPONSE
{candidate_response}
""".strip()

    inputs = judge_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**inputs, max_new_tokens=32)
    text = judge_tokenizer.decode(out[0], skip_special_tokens=True)

    rel = emp = gro = 0
    m = re.search(r"Relevance\s*=\s*(\d+)", text)
    if m: rel = int(m.group(1))
    m = re.search(r"Empathy\s*=\s*(\d+)", text)
    if m: emp = int(m.group(1))
    m = re.search(r"Grounding\s*=\s*(\d+)", text)
    if m: gro = int(m.group(1))

    rel = max(0, min(5, rel))
    emp = max(0, min(5, emp))
    gro = max(0, min(5, gro))
    return rel + emp + gro


# -----------------------------
# Model (4-bit + LoRA)
# -----------------------------
def make_lora_model(base_model_name, lora_r, lora_alpha, lora_dropout):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(base, lora_config)
    return model


# -----------------------------
# Tokenization
# -----------------------------
def tokenize_texts(tokenizer, text_list, max_length):
    enc = tokenizer(text_list, truncation=True, padding="max_length", max_length=max_length)
    enc["labels"] = [ids.copy() for ids in enc["input_ids"]]
    return enc


# -----------------------------
# Main trial
# -----------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--emowoz", required=True, help="Path to emowoz-dialmage.json (e.g., /content/drive/...json)")
    ap.add_argument("--base_model", default="meta-llama/Llama-2-7b-chat-hf")
    ap.add_argument("--judge_model", default="google/flan-t5-base")

    ap.add_argument("--lr", type=float, required=True)
    ap.add_argument("--r", type=int, required=True)
    ap.add_argument("--alpha", type=int, required=True)
    ap.add_argument("--dropout", type=float, required=True)
    ap.add_argument("--epochs", type=int, default=1)

    ap.add_argument("--max_len", type=int, default=512)
    ap.add_argument("--train_cap", type=int, default=400, help="Cap train samples for fast trials")
    ap.add_argument("--max_val_samples", type=int, default=30)

    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--out", required=True, help="Output directory for this trial")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)

    # Reproducibility
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load data
    with open(args.emowoz, "r", encoding="utf-8") as f:
        emowoz_json = json.load(f)

    samples = build_emowoz_samples(emowoz_json, max_dialogs=None)
    random.shuffle(samples)

    val_size = max(50, int(0.1 * len(samples)))
    val_samples = samples[:val_size]
    train_samples = samples[val_size:]

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.base_model)
    tokenizer.pad_token = tokenizer.eos_token

    # Build datasets
    train_ds = Dataset.from_dict({"text": train_samples})
    val_ds = Dataset.from_dict({"text": val_samples})

    train_tok = train_ds.map(
        lambda x: tokenize_texts(tokenizer, x["text"], args.max_len),
        batched=True,
        remove_columns=["text"],
    )
    val_tok = val_ds.map(
        lambda x: tokenize_texts(tokenizer, x["text"], args.max_len),
        batched=True,
        remove_columns=["text"],
    )

    # Cap training set for faster tuning
    train_tok = train_tok.select(range(min(args.train_cap, len(train_tok))))

    # Precompute val prompts (no gold response)
    val_texts = val_samples
    val_prompts = [strip_gold_response(t) for t in val_texts]

    # Judge
    judge_tokenizer, judge_model = make_judge(args.judge_model, device)

    # Model (4-bit + LoRA)
    model = make_lora_model(args.base_model, args.r, args.alpha, args.dropout)

    # Train
    training_args = TrainingArguments(
        output_dir=os.path.join(args.out, "hf_out"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        fp16=True,
        logging_steps=20,
        save_steps=999999,  # no intermediate saves during trials
        save_total_limit=1,
        report_to="none",
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_tok)
    trainer.train()

    # Eval (subset, using judge)
    model.eval()
    pick = list(range(len(val_prompts)))
    random.shuffle(pick)
    pick = pick[: min(args.max_val_samples, len(val_prompts))]

    scores = []
    for idx in pick:
        prompt = val_prompts[idx]
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=80, temperature=0.7, do_sample=True)

        decoded = tokenizer.decode(out[0], skip_special_tokens=True)
        cand = decoded.split("### Response:")[-1].strip() if "### Response:" in decoded else decoded.strip()
        scores.append(judge_score(judge_tokenizer, judge_model, prompt, cand))

    avg_score = float(np.mean(scores)) if scores else 0.0

    # Save result for controller
    result = {
        "avg_judge_score_0_15": avg_score,
        "params": {"lr": args.lr, "r": args.r, "alpha": args.alpha, "dropout": args.dropout, "epochs": args.epochs},
        "meta": {"max_val_samples": args.max_val_samples, "train_cap": args.train_cap, "max_len": args.max_len},
    }
    with open(os.path.join(args.out, "result.json"), "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    # Optional: save adapter for this trial (comment out if you want only result.json)
    # model.save_pretrained(os.path.join(args.out, "lora_adapter"))
    # tokenizer.save_pretrained(os.path.join(args.out, "lora_adapter"))

    # Clean up VRAM explicitly (then process exits -> OS frees everything)
    del trainer, model, judge_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("✅ Trial done. Avg judge score:", avg_score)
    print("Saved:", os.path.join(args.out, "result.json"))


if __name__ == "__main__":
    main()


In [ ]:
# final_train.py
# Train ONE final model using the BEST hyperparameters and SAVE to OUTPUT_DIR.
#
# Example:
# python final_train.py --emowoz "/content/drive/MyDrive/AI/emowoz-dialmage.json" \
#   --base_model "meta-llama/Llama-2-7b-chat-hf" \
#   --lr 1e-4 --r 8 --alpha 16 --dropout 0.05 --epochs 1 \
#   --out "./llama2-emowoz-best"
%%writefile final_train.py
# ...final code only...

import os, json, re, random, argparse, gc
from collections import Counter
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model


# -----------------------------
# Data helpers (same as trials)
# -----------------------------
def extract_emotion_id(emotion_list):
    if not emotion_list:
        return None
    for e in emotion_list:
        if isinstance(e, dict) and "emotion" in e:
            return int(e["emotion"])
    ann = []
    for e in emotion_list:
        if isinstance(e, dict) and "annotation" in e:
            ann.append(int(e["annotation"]))
    if ann:
        return Counter(ann).most_common(1)[0][0]
    return None


def dialog_act_to_facts(dialog_act):
    facts = []
    if not dialog_act:
        return ""
    for act, pairs in dialog_act.items():
        for p in pairs:
            if isinstance(p, (list, tuple)) and len(p) >= 2:
                slot, val = p[0], p[1]
                facts.append(f"{act}.{slot} = {val}")
    return "\n".join(facts)


def build_emowoz_samples(emowoz_json, max_dialogs=None):
    samples = []
    dialog_ids = list(emowoz_json.keys())
    if max_dialogs:
        dialog_ids = dialog_ids[:max_dialogs]

    for did in dialog_ids:
        log = emowoz_json[did]["log"]
        history = []
        accumulated_facts = []
        last_user_emotion = None

        for turn in log:
            text = str(turn.get("text", "")).strip()
            dialog_act = turn.get("dialog_act", {}) or {}
            emo_list = turn.get("emotion", []) or []
            if not text:
                continue

            has_act = bool(dialog_act)
            has_emo = bool(emo_list)

            if has_act:
                role = "assistant"
            elif has_emo:
                role = "user"
            else:
                role = "assistant" if (len(history) > 0 and history[-1].startswith("User:")) else "user"

            if role == "user":
                last_user_emotion = extract_emotion_id(emo_list)
                history.append(f"User: {text}")
            else:
                new_facts = dialog_act_to_facts(dialog_act)
                if new_facts:
                    accumulated_facts.append(new_facts)

                sample = f"""### Instruction:
You are an emotionally intelligent, professional task-oriented dialogue agent.
You must be empathetic and calm, and you must NOT invent facts.
Use only the provided facts when stating addresses, phone numbers, times, prices, etc.

### User Emotion:
{str(last_user_emotion) if last_user_emotion is not None else "unknown"}

### Provided Facts:
{chr(10).join(accumulated_facts) if accumulated_facts else "none"}

### Context:
{chr(10).join(history)}

### Response:
{text}
"""
                samples.append(sample.strip())
                history.append(f"Assistant: {text}")
    return samples


def make_lora_model(base_model_name, lora_r, lora_alpha, lora_dropout):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = get_peft_model(base, lora_config)
    return model


def tokenize_texts(tokenizer, text_list, max_length):
    enc = tokenizer(text_list, truncation=True, padding="max_length", max_length=max_length)
    enc["labels"] = [ids.copy() for ids in enc["input_ids"]]
    return enc


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--emowoz", required=True, help="Path to emowoz-dialmage.json")
    ap.add_argument("--base_model", default="meta-llama/Llama-2-7b-chat-hf")

    ap.add_argument("--lr", type=float, required=True)
    ap.add_argument("--r", type=int, required=True)
    ap.add_argument("--alpha", type=int, required=True)
    ap.add_argument("--dropout", type=float, required=True)
    ap.add_argument("--epochs", type=int, default=1)

    ap.add_argument("--max_len", type=int, default=512)
    ap.add_argument("--seed", type=int, default=42)

    ap.add_argument("--out", required=True, help="OUTPUT_DIR to save final adapter + tokenizer")
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)

    # Seeds
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    # Load dataset
    with open(args.emowoz, "r", encoding="utf-8") as f:
        emowoz_json = json.load(f)

    samples = build_emowoz_samples(emowoz_json, max_dialogs=None)
    random.shuffle(samples)

    # Use a normal split (you can also choose to train on ALL samples)
    val_size = max(50, int(0.1 * len(samples)))
    train_samples = samples[val_size:]  # final train uses the remaining 90%

    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.base_model)
    tokenizer.pad_token = tokenizer.eos_token

    # Dataset + tokenize
    train_ds = Dataset.from_dict({"text": train_samples})
    train_tok = train_ds.map(
        lambda x: tokenize_texts(tokenizer, x["text"], args.max_len),
        batched=True,
        remove_columns=["text"],
    )

    # Build model
    model = make_lora_model(args.base_model, args.r, args.alpha, args.dropout)

    # Train args
    train_args = TrainingArguments(
        output_dir=os.path.join(args.out, "hf_out"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        fp16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=1,
        report_to="none",
    )

    trainer = Trainer(model=model, args=train_args, train_dataset=train_tok)
    trainer.train()

    # SAVE: adapter + tokenizer
    model.save_pretrained(args.out)
    tokenizer.save_pretrained(args.out)

    # Also save the hyperparams used (nice for future)
    with open(os.path.join(args.out, "best_params_used.json"), "w", encoding="utf-8") as f:
        json.dump(
            {"lr": args.lr, "r": args.r, "alpha": args.alpha, "dropout": args.dropout, "epochs": args.epochs},
            f,
            indent=2,
        )

    # Cleanup
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("✅ Final training complete. Saved to:", args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile controller.py
# ...controller code only...
import os
import json
import random
import subprocess
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
EMOWOZ_PATH = "/content/drive/MyDrive/AI/emowoz-dialmage.json"
TRIAL_SCRIPT = "train_one_trial.py"

# ✅ NEW: final training script name
FINAL_SCRIPT = "final_train.py"   # <-- NEW

BASE_OUT_DIR = "runs"
FINAL_OUT_DIR = "./llama2-emowoz-best"  # <-- NEW (your requested final folder)

search_space = {
    "lr": [5e-5, 1e-4, 2e-4],
    "r": [4, 8, 16],
    "alpha": [8, 16, 32],
    "dropout": [0.0, 0.05],
    "epochs": [1]
}

NUM_TRIALS = 6

os.makedirs(BASE_OUT_DIR, exist_ok=True)

results = []
best = {"score": -1, "params": None}

# -----------------------------
# TUNING LOOP (subprocess trials)
# -----------------------------
for trial_id in range(NUM_TRIALS):

    params = {
        "lr": random.choice(search_space["lr"]),
        "r": random.choice(search_space["r"]),
        "alpha": random.choice(search_space["alpha"]),
        "dropout": random.choice(search_space["dropout"]),
        "epochs": random.choice(search_space["epochs"]),
    }

    outdir = os.path.join(
        BASE_OUT_DIR,
        f"trial_{trial_id}_lr{params['lr']}_r{params['r']}_a{params['alpha']}_d{params['dropout']}"
    )
    os.makedirs(outdir, exist_ok=True)

    cmd = [
        "python", TRIAL_SCRIPT,
        "--emowoz", EMOWOZ_PATH,
        "--lr", str(params["lr"]),
        "--r", str(params["r"]),
        "--alpha", str(params["alpha"]),
        "--dropout", str(params["dropout"]),
        "--epochs", str(params["epochs"]),
        "--out", outdir
    ]

    print("\n==============================")
    print("Running trial:", trial_id)
    print("Params:", params)
    print("Command:", " ".join(cmd))

    subprocess.run(cmd, check=True)

    result_path = os.path.join(outdir, "result.json")
    if not os.path.exists(result_path):
        print("⚠️ result.json missing, skipping this trial")
        continue

    with open(result_path, "r", encoding="utf-8") as f:
        trial_result = json.load(f)

    score = trial_result.get("avg_judge_score_0_15", None)
    if score is None:
        print("⚠️ score missing in result.json, skipping")
        continue

    results.append({
        "trial_id": trial_id,
        "score": score,
        **params
    })

    if score > best["score"]:
        best["score"] = score
        best["params"] = params

# -----------------------------
# SUMMARY
# -----------------------------
df = pd.DataFrame(results)
if not df.empty:
    df = df.sort_values("score", ascending=False)
    df.to_csv(os.path.join(BASE_OUT_DIR, "all_results.csv"), index=False)

print("\n==============================")
print("BEST FOUND:")
print("Score:", best["score"])
print("Params:", best["params"])
print("==============================")

# ✅ NEW: save best params to a file (so you can reuse later)
if best["params"] is not None:
    with open(os.path.join(BASE_OUT_DIR, "best_params.json"), "w", encoding="utf-8") as f:
        json.dump(best["params"], f, indent=2)
    print("✅ Saved best params to:", os.path.join(BASE_OUT_DIR, "best_params.json"))

# ✅ NEW: FINAL TRAINING STEP (runs as ONE more subprocess)
# This is the missing part you were asking about.
if best["params"] is not None:
    os.makedirs(FINAL_OUT_DIR, exist_ok=True)

    final_cmd = [
        "python", FINAL_SCRIPT,
        "--emowoz", EMOWOZ_PATH,
        "--lr", str(best["params"]["lr"]),
        "--r", str(best["params"]["r"]),
        "--alpha", str(best["params"]["alpha"]),
        "--dropout", str(best["params"]["dropout"]),
        "--epochs", str(best["params"]["epochs"]),
        "--out", FINAL_OUT_DIR
    ]

    print("\n==============================")
    print("RUNNING FINAL TRAINING (BEST PARAMS)")
    print("Output dir:", FINAL_OUT_DIR)
    print("Command:", " ".join(final_cmd))

    subprocess.run(final_cmd, check=True)

    print("✅ Final model saved to:", FINAL_OUT_DIR)
else:
    print("No best params found (no successful trials), skipping final training.")


In [ ]:
!python controller.py
